In [0]:
# 1. Cargar tablas de productos desde PostgreSQL
df_prod = read_pg("public.producto")
df_cat = read_pg("public.categoria_producto")

df_productos = (
    df_prod.alias("p")
    .join(df_cat.alias("cp"), F.col("p.id_categoria") == F.col("cp.id_categoria"), "inner")
    .select(
        F.col("p.id_producto").alias("id_producto"),
        F.col("p.sku").alias("sku"),
        F.col("p.nombre_producto").alias("nombre_producto"),
        F.col("p.precio_unitario_referencia").alias("precio_unitario_referencia"),
        F.col("cp.nombre_categoria").alias("nombre_categoria")
    )
)

# 2. Concatenar texto de búsqueda y ampliar expresiones regulares para mayor cobertura
texto_busqueda = F.lower(F.concat_ws(" ", F.col("nombre_categoria"), F.col("nombre_producto")))

df_productos_mapped = df_productos.withColumn(
    "categoria_externa",
    F.when(texto_busqueda.rlike("arroz|rice|grano|abarrote|semilla|harina|cereal"), F.lit("RICE"))
     .when(texto_busqueda.rlike("azucar|azúcar|sugar|dulce|endulzante|panela|miel"), F.lit("SUGAR"))
     .when(texto_busqueda.rlike("leche|lacte|lácte|milk|dairy|queso|crema|yogurt|mantequilla"), F.lit("MILK"))
)

# 3. Preparación de la tabla externa
df_ext_raw = spark.table(T_EXT_RAW)
for c in df_ext_raw.columns:
    df_ext_raw = df_ext_raw.withColumnRenamed(c, c.strip())

col_categ = "categ" if "categ" in df_ext_raw.columns else ("category" if "category" in df_ext_raw.columns else df_ext_raw.columns[-1])

df_ext = df_ext_raw.withColumn("precio_externo", F.col("prices").cast("double"))

# 4. Generar Benchmark con conversión explícita PLN a USD (Tasa ref: 1 PLN = 0.25 USD)
PLN_TO_USD_RATE = 0.25

df_benchmark = (
    df_productos_mapped.alias("pm")
    .filter(F.col("pm.categoria_externa").isNotNull())
    .join(df_ext.alias("e"), F.col("pm.categoria_externa") == F.col(f"e.{col_categ}"), "inner")
    .groupBy("pm.id_producto", "pm.sku", "pm.nombre_producto", "pm.precio_unitario_referencia", "pm.categoria_externa")
    .agg(
        F.avg("e.precio_externo").alias("precio_promedio_externo_pln"),
        F.count("*").alias("muestras_externas")
    )
    .withColumn("moneda_externa", F.lit("PLN"))
    .withColumn("tasa_cambio_pln_usd", F.lit(PLN_TO_USD_RATE))
    .withColumn("precio_promedio_externo_usd", F.round(F.col("precio_promedio_externo_pln") * F.lit(PLN_TO_USD_RATE), 2))
    .withColumn("generado_at", F.current_timestamp())
)

df_benchmark.write.mode("overwrite").saveAsTable(T_BENCHMARK)
print(f"Productos con benchmark externo encontrado: {spark.table(T_BENCHMARK).count()}")
display(spark.table(T_BENCHMARK))